# Alpamayo 1.5: Stable Two-Stage Cached Sliding Analysis

This notebook uses a two-stage analysis flow for better stability:

1. Dense sliding pass: compute only numeric metrics.
2. Worst-K replay pass: rerun only the hardest windows to display Front_camera images, full CoT, and trajectory plots.

The clip is preloaded into memory once so repeated window access is faster.

In [ ]:
import os
import sys
import gc
import time
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    preload_offline_clip_to_cache,
    load_offline_dataset_from_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5

In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)

# ===== Dense sliding config =====
SLIDING_START_T0_SOD = 175490.23
SLIDING_END_T0_SOD = 175506.23
SLIDING_STEP_SEC = 1.0

# ===== Eval-side correction =====
EVAL_XY_ROTATION_DEG = -90.0

# ===== Worst-case replay =====
WORST_K = 8
WORST_SORT_BY = 'final_point_error_m'   # or 'minADE_m'
SHOW_TOP_N_WORST = 6
FRONT_CAMERA_INDEX = 1

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('COSMOS_PROCESSOR_PATH =', COSMOS_PROCESSOR_PATH)
print('SLIDING_START_T0_SOD =', SLIDING_START_T0_SOD)
print('SLIDING_END_T0_SOD =', SLIDING_END_T0_SOD)
print('SLIDING_STEP_SEC =', SLIDING_STEP_SEC)
print('WORST_K =', WORST_K)
print('WORST_SORT_BY =', WORST_SORT_BY)
print('SHOW_TOP_N_WORST =', SHOW_TOP_N_WORST)

### Helper functions

In [ ]:
def wrap_to_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def wrap_to_180_deg(x):
    return (x + 180.0) % 360.0 - 180.0

def rotate_xy(xy, angle_deg):
    rad = np.deg2rad(angle_deg)
    c = np.cos(rad)
    s = np.sin(rad)
    R = np.array([[c, -s], [s, c]], dtype=np.float64)
    return xy @ R.T

def rotate_xyz_xy_plane(xyz, angle_deg):
    xyz = xyz.copy()
    xyz[:, :2] = rotate_xy(xyz[:, :2], angle_deg)
    return xyz

def global_motion_yaw_deg(xyz):
    xy = xyz[:, :2]
    disp = xy[-1] - xy[0]
    if np.linalg.norm(disp) < 1e-6:
        return np.nan
    return float(np.rad2deg(np.arctan2(disp[1], disp[0])))

def mean_speed_mps(xyz, dt):
    xy = xyz[:, :2]
    dxy = xy[1:] - xy[:-1]
    step_dist = np.linalg.norm(dxy, axis=1)
    if len(step_dist) == 0:
        return 0.0
    return float(step_dist.mean() / dt)

def yaw_from_rot_plus_x_deg(rot_mats):
    return np.rad2deg(np.arctan2(rot_mats[:, 1, 0], rot_mats[:, 0, 0]))

def history_consistency_summary(hist_xyz_raw, hist_rot, dt):
    dxy = hist_xyz_raw[1:, :2] - hist_xyz_raw[:-1, :2]
    if len(dxy) == 0:
        return {
            'mean_abs_yaw_minus_dxy_deg': np.nan,
            'last5_mean_abs_yaw_minus_dxy_deg': np.nan,
        }

    dxy_yaw_deg = np.rad2deg(np.arctan2(dxy[:, 1], dxy[:, 0]))
    rot_yaw_deg = yaw_from_rot_plus_x_deg(hist_rot)[1:]
    yaw_minus_dxy_deg = wrap_to_180_deg(rot_yaw_deg - dxy_yaw_deg)
    abs_err = np.abs(yaw_minus_dxy_deg)

    return {
        'mean_abs_yaw_minus_dxy_deg': float(abs_err.mean()),
        'last5_mean_abs_yaw_minus_dxy_deg': float(abs_err[-5:].mean()) if len(abs_err) >= 5 else float(abs_err.mean()),
    }

def summarize_main_metrics(gt_xyz, pred_xyz_np, dt):
    gt_xy = gt_xyz[:, :2].T
    pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)

    diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
    best_idx = int(diff.argmin())
    min_ade = float(diff.min())
    pred_best_xyz = pred_xyz_np[best_idx]

    gt_final_xy = gt_xyz[-1, :2]
    pred_final_xy = pred_best_xyz[-1, :2]
    final_point_error = float(np.linalg.norm(pred_final_xy - gt_final_xy))

    gt_mean_speed = mean_speed_mps(gt_xyz, dt)
    pred_mean_speed = mean_speed_mps(pred_best_xyz, dt)
    speed_error = float(pred_mean_speed - gt_mean_speed)

    gt_yaw = global_motion_yaw_deg(gt_xyz)
    pred_yaw = global_motion_yaw_deg(pred_best_xyz)
    if np.isfinite(gt_yaw) and np.isfinite(pred_yaw):
        yaw_error = float(np.rad2deg(wrap_to_pi(np.deg2rad(pred_yaw - gt_yaw))))
    else:
        yaw_error = np.nan

    return {
        'best_sample_idx': best_idx,
        'minADE_m': min_ade,
        'final_point_error_m': final_point_error,
        'gt_mean_speed_mps': gt_mean_speed,
        'pred_mean_speed_mps': pred_mean_speed,
        'speed_error_mps': speed_error,
        'gt_yaw_deg': gt_yaw,
        'pred_yaw_deg': pred_yaw,
        'yaw_error_deg': yaw_error,
        'pred_best_xyz': pred_best_xyz,
    }

def compute_t0_grid(start_sod, end_sod, step_sec):
    n = int(np.floor((end_sod - start_sod) / step_sec)) + 1
    return start_sod + np.arange(n, dtype=np.float64) * step_sec

def _compute_adaptive_xy_limits(*arrays, min_span=20.0, pad_ratio=0.12, pad_min=2.0):
    pts = [np.array([[0.0, 0.0]], dtype=np.float32)]
    for arr in arrays:
        if arr is None:
            continue
        pts.append(arr[:, :2])
    pts = np.concatenate(pts, axis=0)
    xmin, ymin = pts.min(axis=0)
    xmax, ymax = pts.max(axis=0)
    xspan = xmax - xmin
    yspan = ymax - ymin
    span = max(float(xspan), float(yspan), float(min_span))
    xcenter = 0.5 * (xmin + xmax)
    ycenter = 0.5 * (ymin + ymax)
    pad = max(span * pad_ratio, pad_min)
    half = 0.5 * span + pad
    return (xcenter - half, xcenter + half), (ycenter - half, ycenter + half)

def extract_front_camera_frames(data, front_camera_index=1):
    camera_indices = data['camera_indices'].cpu().numpy().tolist()
    if front_camera_index not in camera_indices:
        return None, None, None
    cam_pos = camera_indices.index(front_camera_index)
    frames = data['image_frames'][cam_pos].cpu().numpy()
    req_idx = data['video_frame_indices'][cam_pos].cpu().numpy().tolist()
    act_idx = data['actual_video_frame_indices'][cam_pos].cpu().numpy().tolist()
    return frames, req_idx, act_idx

def show_front_camera_frames(frames, req_idx, act_idx, title_prefix='Front_camera'):
    if frames is None:
        print('Front_camera not found in this sample.')
        return
    num_frames = frames.shape[0]
    fig, axes = plt.subplots(1, num_frames, figsize=(4 * num_frames, 3.5))
    if num_frames == 1:
        axes = [axes]
    for t_idx in range(num_frames):
        ax = axes[t_idx]
        frame = np.transpose(frames[t_idx], (1, 2, 0))
        ax.imshow(frame)
        ax.axis('off')
        ax.set_title(f'{title_prefix}\nt={t_idx}, req={req_idx[t_idx]}, actual={act_idx[t_idx]}')
    plt.tight_layout()
    plt.show()

def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### Preload clip cache

In [ ]:
preload_start = time.time()
clip_cache = preload_offline_clip_to_cache(
    clip_dir=CLIP_DIR,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
)
preload_elapsed = time.time() - preload_start
print(f'Preload done in {preload_elapsed:.2f}s')

### Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')

## Stage 1: Dense sliding numeric pass only

In [ ]:
t0_grid = compute_t0_grid(SLIDING_START_T0_SOD, SLIDING_END_T0_SOD, SLIDING_STEP_SEC)
print('num_windows =', len(t0_grid))

rows = []
global_start_time = time.time()

for idx, t0_sod in enumerate(t0_grid):
    iter_start = time.time()
    try:
        data = load_offline_dataset_from_cache(
            clip_cache=clip_cache,
            t0_sod=float(t0_sod),
            num_history_steps=NUM_HISTORY_STEPS,
            num_future_steps=NUM_FUTURE_STEPS,
            time_step=TIME_STEP,
            num_frames=NUM_FRAMES,
            image_size=IMAGE_SIZE,
            debug=False,
        )

        hist_xyz_raw = data['ego_history_xyz'].cpu().numpy()[0, 0, :, :]
        hist_rot = data['ego_history_rot'].cpu().numpy()[0, 0, :, :, :]
        gt_xyz_raw = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]
        gt_xyz_plot = rotate_xyz_xy_plane(gt_xyz_raw, EVAL_XY_ROTATION_DEG)

        messages = helper.create_message(
            frames=data['image_frames'].flatten(0, 1),
            camera_indices=data['camera_indices'],
        )
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            continue_final_message=True,
            return_dict=True,
            return_tensors='pt',
        )

        model_inputs = {
            'tokenized_data': inputs,
            'ego_history_xyz': data['ego_history_xyz'],
            'ego_history_rot': data['ego_history_rot'],
        }
        model_inputs = helper.to_device(model_inputs, DEVICE)

        if DEVICE.startswith('cuda'):
            autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
        else:
            autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

        with torch.inference_mode():
            with autocast_context:
                pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
                    data=model_inputs,
                    top_p=TOP_P,
                    temperature=TEMPERATURE,
                    num_traj_samples=NUM_TRAJ_SAMPLES,
                    max_generation_length=MAX_GENERATION_LENGTH,
                    return_extra=False,
                )

        pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]
        hist_diag = history_consistency_summary(hist_xyz_raw, hist_rot, TIME_STEP)
        metrics = summarize_main_metrics(gt_xyz=gt_xyz_plot, pred_xyz_np=pred_xyz_np, dt=TIME_STEP)

        rows.append({
            'window_idx': idx,
            't0_sod': float(t0_sod),
            'best_sample_idx': metrics['best_sample_idx'],
            'minADE_m': metrics['minADE_m'],
            'final_point_error_m': metrics['final_point_error_m'],
            'gt_mean_speed_mps': metrics['gt_mean_speed_mps'],
            'pred_mean_speed_mps': metrics['pred_mean_speed_mps'],
            'speed_error_mps': metrics['speed_error_mps'],
            'gt_yaw_deg': metrics['gt_yaw_deg'],
            'pred_yaw_deg': metrics['pred_yaw_deg'],
            'yaw_error_deg': metrics['yaw_error_deg'],
            'mean_abs_yaw_minus_dxy_deg': hist_diag['mean_abs_yaw_minus_dxy_deg'],
            'last5_mean_abs_yaw_minus_dxy_deg': hist_diag['last5_mean_abs_yaw_minus_dxy_deg'],
            'status': 'ok',
            'error': '',
            'elapsed_sec': time.time() - iter_start,
        })

        print(f'[{idx+1}/{len(t0_grid)}] t0={t0_sod:.2f} ok | minADE={metrics["minADE_m"]:.3f} | final={metrics["final_point_error_m"]:.3f} | sec={rows[-1]["elapsed_sec"]:.2f}')

        del data, messages, inputs, model_inputs, pred_xyz, pred_rot, pred_xyz_np
        cleanup_torch()

    except Exception as e:
        rows.append({
            'window_idx': idx,
            't0_sod': float(t0_sod),
            'best_sample_idx': np.nan,
            'minADE_m': np.nan,
            'final_point_error_m': np.nan,
            'gt_mean_speed_mps': np.nan,
            'pred_mean_speed_mps': np.nan,
            'speed_error_mps': np.nan,
            'gt_yaw_deg': np.nan,
            'pred_yaw_deg': np.nan,
            'yaw_error_deg': np.nan,
            'mean_abs_yaw_minus_dxy_deg': np.nan,
            'last5_mean_abs_yaw_minus_dxy_deg': np.nan,
            'status': 'error',
            'error': repr(e),
            'elapsed_sec': time.time() - iter_start,
        })
        print(f'[{idx+1}/{len(t0_grid)}] t0={t0_sod:.2f} ERROR | {repr(e)}')
        cleanup_torch()

stage1_elapsed = time.time() - global_start_time
df_results = pd.DataFrame(rows)
df_ok = df_results[df_results['status'] == 'ok'].reset_index(drop=True)

print('\n[Stage 1 results]')
display(df_results)
print(f'Stage 1 elapsed sec = {stage1_elapsed:.2f}')
print(f'Avg sec / window = {df_results["elapsed_sec"].mean():.2f}')

### Stage 1 summary and worst-K mining

In [ ]:
if len(df_ok) == 0:
    raise RuntimeError('No successful windows. Please inspect df_results.')

summary = pd.DataFrame([{
    'num_windows_total': int(len(df_results)),
    'num_windows_ok': int(len(df_ok)),
    'success_rate': float(len(df_ok) / len(df_results)),
    'minADE_mean_m': float(df_ok['minADE_m'].mean()),
    'minADE_median_m': float(df_ok['minADE_m'].median()),
    'minADE_p90_m': float(df_ok['minADE_m'].quantile(0.90)),
    'final_error_mean_m': float(df_ok['final_point_error_m'].mean()),
    'final_error_median_m': float(df_ok['final_point_error_m'].median()),
    'final_error_p90_m': float(df_ok['final_point_error_m'].quantile(0.90)),
    'abs_speed_error_mean_mps': float(df_ok['speed_error_mps'].abs().mean()),
    'abs_yaw_error_mean_deg': float(df_ok['yaw_error_deg'].abs().mean()),
    'hist_consistency_mean_deg': float(df_ok['mean_abs_yaw_minus_dxy_deg'].mean()),
    'avg_sec_per_window': float(df_results['elapsed_sec'].mean()),
}])

df_worst = df_ok.sort_values(WORST_SORT_BY, ascending=False).head(WORST_K).reset_index(drop=True)

print('[Summary]')
display(summary)

print(f'[Worst-{WORST_K} by {WORST_SORT_BY}]')
display(df_worst)

### Metric trends over time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

axes[0].plot(df_ok['t0_sod'], df_ok['minADE_m'], 'o-', label='minADE_m')
axes[0].set_ylabel('minADE (m)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(df_ok['t0_sod'], df_ok['final_point_error_m'], 'o-', label='final_point_error_m', color='tab:orange')
axes[1].set_ylabel('Final Error (m)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(df_ok['t0_sod'], df_ok['gt_mean_speed_mps'], 'o-', label='GT mean speed')
axes[2].plot(df_ok['t0_sod'], df_ok['pred_mean_speed_mps'], 'o-', label='Pred mean speed')
axes[2].set_xlabel('t0_sod')
axes[2].set_ylabel('Speed (m/s)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

## Stage 2: Replay worst windows with Front_camera + full CoT

In [ ]:
num_preview = min(SHOW_TOP_N_WORST, len(df_worst))

for rank in range(num_preview):
    row = df_worst.iloc[rank]
    t0_sod = float(row['t0_sod'])
    print('=' * 120)
    print(f'Worst rank {rank+1}/{num_preview} | t0={t0_sod:.2f}')

    replay_start = time.time()
    data = load_offline_dataset_from_cache(
        clip_cache=clip_cache,
        t0_sod=t0_sod,
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        image_size=IMAGE_SIZE,
        debug=False,
    )

    hist_xyz_raw = data['ego_history_xyz'].cpu().numpy()[0, 0, :, :]
    gt_xyz_raw = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]
    hist_xyz_plot = rotate_xyz_xy_plane(hist_xyz_raw, EVAL_XY_ROTATION_DEG)
    gt_xyz_plot = rotate_xyz_xy_plane(gt_xyz_raw, EVAL_XY_ROTATION_DEG)

    messages = helper.create_message(
        frames=data['image_frames'].flatten(0, 1),
        camera_indices=data['camera_indices'],
    )
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        continue_final_message=True,
        return_dict=True,
        return_tensors='pt',
    )
    model_inputs = {
        'tokenized_data': inputs,
        'ego_history_xyz': data['ego_history_xyz'],
        'ego_history_rot': data['ego_history_rot'],
    }
    model_inputs = helper.to_device(model_inputs, DEVICE)

    if DEVICE.startswith('cuda'):
        autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
    else:
        autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

    with torch.inference_mode():
        with autocast_context:
            pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
                data=model_inputs,
                top_p=TOP_P,
                temperature=TEMPERATURE,
                num_traj_samples=NUM_TRAJ_SAMPLES,
                max_generation_length=MAX_GENERATION_LENGTH,
                return_extra=True,
            )

    pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]
    metrics = summarize_main_metrics(gt_xyz=gt_xyz_plot, pred_xyz_np=pred_xyz_np, dt=TIME_STEP)
    cot_value = extra['cot'][0]
    if hasattr(cot_value, 'tolist'):
        cot_value = cot_value.tolist()
    cot_text = str(cot_value)

    front_frames, front_req_idx, front_act_idx = extract_front_camera_frames(data, FRONT_CAMERA_INDEX)

    display(pd.DataFrame([{
        't0_sod': t0_sod,
        'minADE_m': metrics['minADE_m'],
        'final_point_error_m': metrics['final_point_error_m'],
        'gt_mean_speed_mps': metrics['gt_mean_speed_mps'],
        'pred_mean_speed_mps': metrics['pred_mean_speed_mps'],
        'speed_error_mps': metrics['speed_error_mps'],
        'gt_yaw_deg': metrics['gt_yaw_deg'],
        'pred_yaw_deg': metrics['pred_yaw_deg'],
        'yaw_error_deg': metrics['yaw_error_deg'],
        'replay_elapsed_sec': time.time() - replay_start,
    }]))

    print('[Full CoT]')
    print(cot_text)

    print('\n[Sampled Front_camera frames]')
    show_front_camera_frames(front_frames, front_req_idx, front_act_idx)

    xlim, ylim = _compute_adaptive_xy_limits(hist_xyz_plot, gt_xyz_plot, metrics['pred_best_xyz'])
    plt.figure(figsize=(6, 6))
    plt.plot(hist_xyz_plot[:, 0], hist_xyz_plot[:, 1], 'b-o', linewidth=2, markersize=3, label='History')
    plt.plot(gt_xyz_plot[:, 0], gt_xyz_plot[:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
    plt.plot(metrics['pred_best_xyz'][:, 0], metrics['pred_best_xyz'][:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred')
    plt.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.grid(True, alpha=0.3)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend()
    plt.tight_layout()
    plt.show()

    del data, messages, inputs, model_inputs, pred_xyz, pred_rot, extra, pred_xyz_np
    cleanup_torch()